# 02 -- Experimentos: H0 vs H1 vs H2 vs H3 bajo condiciones idénticas

Corre las 4 políticas sobre EXACTAMENTE la misma demanda, flota, capacidad y
horizonte -- la única variable es la política. Todas las gráficas son
interactivas (Plotly).

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import yaml

BASE = Path.cwd().parent.parent
sys.path.insert(0, str(BASE / "simulador" / "src"))
sys.path.insert(0, str(BASE / "politica_base" / "src"))
sys.path.insert(0, str(BASE / "heuristicas" / "comun" / "src"))
sys.path.insert(0, str(BASE / "heuristicas" / "h1_reserva" / "src"))
sys.path.insert(0, str(BASE / "heuristicas" / "h2_costo_local" / "src"))
sys.path.insert(0, str(BASE / "heuristicas" / "h3_costo_global" / "src"))

from env import SimuladorBarcosBergen
from politica_base import asignar_flota
from politica_h1 import asignar_flota_h1
from politica_h2 import asignar_flota_h2
from politica_h3 import asignar_flota_h3
import metricas as met

cfg_sim = yaml.safe_load(open(BASE / "simulador" / "config" / "instance.yaml", encoding="utf-8"))
cfg_bb = yaml.safe_load(open(BASE / "bergen-boats" / "config" / "instance.yaml", encoding="utf-8"))
nodos = [n["id"] for n in cfg_bb["nodos_demanda"]]
matriz_tiempos = pd.read_csv(BASE / "bergen-boats" / "02_ruteo_navegable" / "output" / "matriz_tiempos_min.csv", index_col=0)
capacidad = cfg_bb["flota"]["capacidad_pasajeros"]

SEMILLA = 1001
NUM_BARCOS = 8
cfg_escalon = cfg_sim["escalones"]["escalon_dia_10pct"]
hora_ini_min, hora_fin_min = cfg_escalon["horas"][0] * 60, cfg_escalon["horas"][1] * 60

grupos = pd.read_csv(BASE / "politica_base" / "output" / "escalon_dia_10pct" / f"grupos_seed{SEMILLA}.csv")
print(f"semilla={SEMILLA}  num_barcos={NUM_BARCOS}  capacidad={capacidad}  "
      f"horizonte=[{hora_ini_min},{hora_fin_min}] min  paso={cfg_sim['paso_tiempo_min']} min")
print(f"demanda: {len(grupos)} grupos, {grupos['tamano_grupo'].sum()} personas")

semilla=1001  num_barcos=8  capacidad=30  horizonte=[360,1440] min  paso=2 min
demanda: 981 grupos, 8324 personas


In [2]:
POLITICAS_CON_RESERVA = {"H1": asignar_flota_h1, "H2": asignar_flota_h2, "H3": asignar_flota_h3}

def correr(nombre_politica):
    env = SimuladorBarcosBergen(
        grupos_df=grupos, matriz_tiempos=matriz_tiempos, nodos=nodos,
        num_barcos=NUM_BARCOS, capacidad_barco=capacidad,
        nodo_inicial=cfg_bb["flota"]["nodo_inicial"], paso_tiempo_min=cfg_sim["paso_tiempo_min"],
        hora_inicio_min=hora_ini_min, hora_fin_min=hora_fin_min,
        cfg_recompensa=cfg_sim["recompensa"], unidad_demanda=cfg_sim["unidad_demanda"],
    )
    obs, info = env.reset(seed=cfg_sim["semilla"])
    reservas = {}
    while True:
        estado = info["_estado_obj"]
        libres = [b for b in estado.barcos if b.libre]
        if nombre_politica == "H0":
            decisiones = asignar_flota(libres, estado, matriz_tiempos, capacidad, cfg_sim["recompensa"])
        else:
            decisiones, reservas = POLITICAS_CON_RESERVA[nombre_politica](
                libres, estado, matriz_tiempos, capacidad, reservas, cfg_sim["recompensa"])
        accion = np.array([env.codificar_accion_barco(decisiones[b.id]) if b.libre else 0 for b in estado.barcos])
        obs, r, terminated, truncated, info = env.step(accion)
        if truncated or terminated:
            break
    return env

envs = {pol: correr(pol) for pol in ["H0", "H1", "H2", "H3"]}
for pol, env in envs.items():
    cons = met.verificar_conservacion(env)
    print(f"{pol}: conservacion={cons['cuadra']}  ({cons['atendidas']}/{cons['generadas']} atendidas)")

H0: conservacion=True  (8267/8324 atendidas)
H1: conservacion=True  (8227/8324 atendidas)
H2: conservacion=True  (8254/8324 atendidas)
H3: conservacion=True  (8279/8324 atendidas)


In [3]:
filas = []
for pol, env in envs.items():
    glob = met.metricas_globales(env)
    usr = met.metricas_por_usuario(env)
    por_barco = met.metricas_por_barco(env)
    backlog = met.sin_atender_al_final_por_par(env)
    movs = [e for e in env.log_eventos if e["tipo"] == "movimiento"]
    movimientos_vacios = sum(1 for m in movs if m["ocupacion"] == 0)
    viajes = list(met._tiempos_viaje_por_unidad(env).values())
    filas.append({
        "politica": pol,
        "pct_atendidas": round(glob["pct_atendidas"], 2),
        "backlog_final": int(backlog["sin_atender"].sum()) if len(backlog) else 0,
        "espera_media_min": round(glob["espera_media_min"], 2),
        "espera_p50_min": round(usr["espera_min"]["p50"], 2),
        "espera_p95_min": round(usr["espera_min"]["p95"], 2),
        "espera_max_min": round(usr["espera_min"]["max"], 2),
        "viaje_medio_min": round(float(np.mean(viajes)), 2) if viajes else float("nan"),
        "sistema_medio_min": round(glob["sistema_medio_min"], 2),
        "sistema_p95_min": round(usr["sistema_min"]["p95"], 2),
        "sistema_max_min": round(usr["sistema_min"]["max"], 2),
        "movimientos_totales": len(movs),
        "movimientos_vacios": movimientos_vacios,
        "ocupacion_pct_capacidad": round(float(por_barco["ocupacion_media"].mean()) / capacidad * 100, 1),
        "pct_esperando_flota": round(float(por_barco["pct_esperando"].mean()), 1),
    })
tabla = pd.DataFrame(filas).set_index("politica")
tabla

,pct_atendidas,backlog_final,espera_media_min,espera_p50_min,espera_p95_min,espera_max_min,viaje_medio_min,sistema_medio_min,sistema_p95_min,sistema_max_min,movimientos_totales,movimientos_vacios,ocupacion_pct_capacidad,pct_esperando_flota
politica,,,,,,,,,,,,,,
H0,99.32,57,11.40,9.82,28.18,41.57,9.73,21.13,38.54,53.57,870,355,24.6,23.8
H1,98.83,97,12.00,10.50,28.82,50.50,9.72,21.72,38.82,62.50,726,210,24.6,36.4
H2,99.16,70,11.96,10.56,29.46,41.03,9.73,21.69,39.46,53.03,731,215,24.6,36.8
H3,99.46,45,10.37,8.93,25.57,34.93,9.72,20.10,36.68,46.93,709,164,24.6,39.7


## Comparación interactiva

In [4]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

COLOR = {"H0": "#2a78d6", "H1": "#eb6834", "H2": "#1baf7a", "H3": "#008300"}
metricas_barras = [
    ("espera_media_min", "Espera media (min)"), ("espera_p95_min", "Espera P95 (min)"),
    ("espera_max_min", "Espera máxima (min)"), ("sistema_medio_min", "Sistema medio (min)"),
    ("movimientos_totales", "Movimientos totales"), ("ocupacion_pct_capacidad", "Ocupación (% capacidad)"),
]

fig = make_subplots(rows=2, cols=3, subplot_titles=[m[1] for m in metricas_barras])
for idx, (col, titulo) in enumerate(metricas_barras):
    row, coln = idx // 3 + 1, idx % 3 + 1
    fig.add_trace(go.Bar(x=tabla.index, y=tabla[col], marker_color=[COLOR[p] for p in tabla.index],
                          showlegend=False, hovertemplate=f"%{{x}}<br>{titulo}: %{{y}}<extra></extra>"),
                  row=row, col=coln)
fig.update_layout(height=650, width=1100, template="plotly_white",
                   title=f"H0 vs H1 vs H2 vs H3 -- {NUM_BARCOS} barcos, semilla {SEMILLA}")
fig.show()

## Perfil de espera en el tiempo (las 4 políticas)

Reusa `simulador/src/visualizacion.graficar_perfil_espera` -- Plotly
interactivo, zoom/pan nativo, sin duplicar lógica de gráficas.

In [5]:
sys.path.insert(0, str(BASE / "simulador" / "src"))
import visualizacion as viz

fig_esperas = go.Figure()
for pol, env in envs.items():
    f = viz.graficar_perfil_espera(env)
    fig_esperas.add_trace(go.Scatter(x=f.data[0]["x"], y=f.data[0]["y"], mode="lines", name=pol,
                                      line=dict(color=COLOR[pol])))
fig_esperas.update_layout(title="Personas esperando en el tiempo -- H0 vs H1 vs H2 vs H3",
                           xaxis_title="Paso", yaxis_title="Personas esperando", template="plotly_white")
fig_esperas.show()

In [6]:
fig_ocup = go.Figure()
for pol, env in envs.items():
    ocup_media = [sum(f["barcos"][i]["ocupacion"] for i in range(NUM_BARCOS)) for f in env.historial_estados]
    fig_ocup.add_trace(go.Scatter(x=list(range(len(ocup_media))), y=ocup_media, mode="lines", name=pol,
                                   line=dict(color=COLOR[pol])))
fig_ocup.update_layout(title="Ocupación total de la flota en el tiempo -- H0 vs H1 vs H2 vs H3",
                        xaxis_title="Paso", yaxis_title="Personas a bordo (suma de la flota)", template="plotly_white")
fig_ocup.show()

## Interpretación

Con 8 barcos y esta semilla, H3 es consistentemente la mejor de las 4 en
espera media/P95/máxima, con menos movimientos totales -- ver
`03_comparacion_flota.ipynb` para confirmar que el patrón se sostiene en
todo el rango de flota (4-16 barcos).